# 구글 마운트

In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

# full_tracked 로드 (주자상황 체크.ipynb에서 저장한 파일)

In [ ]:
# 처리할 시즌. 한 해만 적으면 예전과 똑같이 그 해만 돈다.
#   YEARS = [2026]                              # ← 평소(현재 시즌만)
YEARS = [2026]
# YEARS = [2026]

# 아래 수동 점검 셀들(등번호 000 채우기 / 개명 반영 / 일정 정리)과 단일 연도 실행이
# 보는 연도. 여러 해를 돌릴 때도 이 값은 마지막 연도로 남는다.
YEAR     = YEARS[-1]
BASE_DIR = f"/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"

import pandas as pd
import os


def load_full(year):
    """full_tracked_{year} 로드.

    parquet 가 있으면 그걸 읽는다 — 통합 체크가 xlsx 와 같은 내용으로 같이 저장한다
    (8만행 기준 11초 -> 0.5초). 아직 검수를 다시 안 돌린 연도는 parquet 가 없어 xlsx 로 간다.
    """
    xlsx    = f"{BASE_DIR}/season_full/full_tracked_{year}.xlsx"
    parquet = f"{BASE_DIR}/season_full/full_tracked_{year}.parquet"
    src = parquet if os.path.exists(parquet) else xlsx
    df = pd.read_parquet(src) if src.endswith(".parquet") else pd.read_excel(src)
    print(f"로드 완료: {src} ({len(df)}행)")
    return df


full = load_full(YEAR)


In [ ]:
# ════════════════════════════════════════════════════════════════
# 몰수경기 등 원본 스코어시트에 아예 없는 경기 결과를 자동으로 보충
#   - "특수경기결과 크롤링.ipynb"가 만든 special_games_{YEAR}.xlsx를 읽어서,
#     원본 크롤러가 놓친(=raw 스코어시트가 아예 없는) 경기의 결과만 채운다.
#     몰수경기 등 실제 타석 진행이 없는 경기라 선수 기록은 없고, game_id 단위
#     컬럼(team1/score1/result1/team2/score2/result2/tournament/league/date)만
#     채운 "결과 전용" 행이다 — batting_totals/pitching_totals 같은 선수 집계
#     에는 안 잡히고, 순위표/일정처럼 game_id 단위로 보는 곳에만 반영된다.
#   - 이미 full에 있는 game_id는 건너뛴다(재실행 대비 중복 방지).
#   - special_games 파일이 아직 없으면(크롤러를 아직 안 돌렸으면) 그냥 건너뛴다.
# ════════════════════════════════════════════════════════════════
def add_special_games(full, year):
    """몰수경기 등 원본 스코어시트에 아예 없는 경기 결과를 full 에 보충해 돌려준다."""
    path = f"{BASE_DIR}/season_full/special_games_{year}.xlsx"
    if not os.path.exists(path):
        print(f"⚠ {path} 가 없습니다 — '특수경기결과 크롤링.ipynb'를 먼저 실행하면 자동으로 채워집니다.")
        return full

    special_games = pd.read_excel(path)
    existing_ids = set(pd.to_numeric(full["game_id"], errors="coerce").dropna().astype(int).tolist())
    special_ids = pd.to_numeric(special_games["game_id"], errors="coerce")
    to_add = special_games[~special_ids.isin(existing_ids) & special_ids.notna()]

    keep_cols = [
        "game_id", "date", "tournament", "league",
        "team1", "score1", "result1", "team2", "score2", "result2",
    ]
    to_add = to_add[[c for c in keep_cols if c in to_add.columns]]

    if to_add.empty:
        print("추가할 특수경기결과 없음(이미 반영돼 있거나 대상 없음)")
        return full

    full = pd.concat([full, to_add], ignore_index=True)
    print(f"특수경기결과 {len(to_add)}건 추가 -> full ({len(full)}행)")
    print(to_add[["game_id", "team1", "score1", "team2", "score2", "league"]].to_string(index=False))
    return full


full = add_special_games(full, YEAR)


# 시즌 통합 파일 export (season_full + person_no)

In [ ]:
# ════════════════════════════════════════════════════════════════
# players_{year}.xlsx 컬럼 매칭 확인 + 팀명 일치 여부 확인
#   - players_{year}.xlsx의 실제 컬럼명과 자동 매칭 결과를 확인한다
#   - 여기서 정해진 name_col/no_col/team_col/person_col을 아래
#     "시즌 통합 파일 export" 셀이 그대로 쓴다
#   - full(경기 박스스코어 team/def_team)과 players의 학교명이 실제로 같은
#     표기인지도 같이 확인한다 (예: "충암고" vs "GD챌린저스BC(U-18)")
#     -> 안 맞아도 export 셀이 선수 명부 겹침으로 자동 연결하므로 참고용이다
# ════════════════════════════════════════════════════════════════
import re as _re

# 실제 컬럼명을 알게 되면 이 후보 목록만 좁혀주면 됨 (제일 앞에 있는 게 우선 매칭)
_NAME_CANDS   = ['player_name', 'name', '선수명', '이름', '성명']
_NO_CANDS     = ['uniform_no', 'back_no', '등번호', 'number', 'no']
_TEAM_CANDS   = ['team', '소속팀', '팀명', '팀', '학교명']
_PERSON_CANDS = ['person_no', 'person_id', 'kbsa_person_no', 'id']


def _first_match(cols, candidates):
    return next((c for c in candidates if c in cols), None)


def players_path(year):
    """그 해 명단 파일 경로. 아래 '등번호 000 채우기' 셀도 이걸 쓴다."""
    return f"{BASE_DIR}/players_by_season/players_{year}.xlsx"


def match_player_columns(year, full=None, verbose=True):
    """players_{year}.xlsx 를 읽고 (players, name_col, no_col, team_col, person_col) 을 돌려준다.

    예전에는 이 셀이 전역 players/name_col/... 을 직접 만들었다. 함수로 바꾼 건
    여러 시즌을 한 번에 돌리기 위해서다 — 맨 아래 일괄 실행 셀이 연도마다 부른다.
    """
    players = pd.read_excel(players_path(year))

    name_col   = _first_match(players.columns, _NAME_CANDS)
    no_col     = _first_match(players.columns, _NO_CANDS)
    team_col   = _first_match(players.columns, _TEAM_CANDS)
    person_col = _first_match(players.columns, _PERSON_CANDS)

    if verbose:
        print(f"players_{year}.xlsx 컬럼 목록: {list(players.columns)}")
        print(f"자동 매칭 결과 -> 이름={name_col}, 등번호={no_col}, 팀={team_col}, person_no={person_col}")

    missing = [label for label, col in [
        ('이름', name_col), ('등번호', no_col), ('팀', team_col), ('person_no', person_col),
    ] if col is None]
    if missing:
        print(f"⚠ 매칭 실패: {missing} 에 해당하는 컬럼을 못 찾았습니다.")
        print("  이 셀 상단의 _NAME_CANDS/_NO_CANDS/_TEAM_CANDS/_PERSON_CANDS 를 실제 컬럼명으로 고치세요.")
    elif verbose:
        print("✅ 4개 컬럼 모두 자동 매칭 성공")

    # ── 팀명 일치 여부(참고) ──────────────────────────────────
    # 안 맞는 팀이 있어도 export 셀이 선수 명부 겹침으로 이어 붙이므로 보통 그대로 둔다.
    # (동산정보산업고 vs 서울동산고, 밀성고 vs TKBC(U19) 같은 개명이 매년 나온다)
    if verbose and full is not None and team_col is not None:
        def _norm(v):
            return v if pd.isna(v) else _re.sub(r'\(.*?\)', '', str(v)).strip()
        game_teams = set(full['team'].dropna().unique()) | set(full['def_team'].dropna().unique())
        raw = set(players[team_col].dropna().unique())
        norm = {_norm(t) for t in raw}
        unmatched = sorted(game_teams - raw - norm)
        print(f"--- 팀명 일치 확인 --- 경기 {len(game_teams)}팀 / 명단 {len(raw)}팀")
        print(f"이름으로 안 맞는 경기 팀 ({len(unmatched)}개): {unmatched[:20]}")

    return players, name_col, no_col, team_col, person_col


players, name_col, no_col, team_col, person_col = match_player_columns(YEAR, full)
# 아래 수동 셀들이 쓰던 전역. 함수로 바꾸면서 사라졌던 것이라 여기서 다시 채워 둔다.
PLAYERS_PATH = players_path(YEAR)


In [ ]:
# ════════════════════════════════════════════════════════════════
# players_{YEAR}.xlsx 등번호 빈 값 -> "000" 일괄 채우기
#   - 고교야구는 등번호가 항상 1~2자리라 "000"은 실제 등번호와 절대 안
#     겹치는 안전한 "등번호 미확인" 표시값이다.
#   - 매칭 실패 자체를 줄여주진 않는다 — 동명이인 매칭이 막히는 건 등번호가
#     비어있어서가 아니라 같은 팀+이름에 서로 다른 person_no가 있어서라,
#     000이든 빈칸이든 그 지점은 동일하다. 그냥 "확인 안 됨"을 명확히
#     표시해서 나중에 빈칸인지 실수로 지워진 건지 헷갈리지 않게 하는 용도.
#   - DRY_RUN=True면 몇 행이 바뀌는지만 보여주고 저장은 안 한다. 확인 후
#     DRY_RUN=False로 바꿔서 다시 실행해야 players 파일에 실제로 저장된다.
# ════════════════════════════════════════════════════════════════
DRY_RUN = True  # 확인 후 False로 바꿔서 재실행 -> 실제 players 파일에 저장

PLAYERS_PATH = players_path(YEAR)   # 위 '컬럼 매칭' 셀의 헬퍼
_players_fresh = pd.read_excel(PLAYERS_PATH)
_blank_mask = _players_fresh[no_col].isna()
n_blank = int(_blank_mask.sum())

print(f"등번호 빈 값: {n_blank}행 / 전체 {len(_players_fresh)}행")
if n_blank:
    print(_players_fresh.loc[_blank_mask, [person_col, name_col, team_col]].head(10).to_string(index=False))

if not DRY_RUN:
    _players_fresh.loc[_blank_mask, no_col] = "000"
    _players_fresh.to_excel(PLAYERS_PATH, index=False)
    print(f"\n저장 완료 -> {PLAYERS_PATH} ({n_blank}행을 '000'으로 채움)")
    print("이제 위쪽 'players 컬럼 매칭 확인' 셀부터 다시 실행해서 players를 새로 불러오세요.")
else:
    print("\n[DRY RUN] 실제로 반영하려면 DRY_RUN = False 로 바꾸고 이 셀을 다시 실행하세요.")


In [ ]:
import subprocess as _subprocess, sys as _sys
_subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=False)

# ════════════════════════════════════════════════════════════════
# 시즌 통합 파일 export: season_full_{year}.xlsx
#   - batter/pitcher 원본 표기가 "이름(등번호)" 형태이므로 이름/등번호를 분리한다
#     (일부는 "이름()" 처럼 등번호가 비어있는 경우도 있어 이것도 처리한다)
#   - pitcher 컬럼을 (game_id, def_team) 기준으로 ffill 하여 매 행마다 채운다
#     (원본은 등판 시작 행에만 값이 있고 나머지는 NaN)
#   - 바로 위 진단 셀에서 확인한 players/name_col/no_col/team_col/person_col을
#     그대로 재사용해 person_no(KBSA 선수 고유 ID)를 join한다
#   - person_no 매칭은 5단계 + 보정 2개다. 2020~2025 를 붙여 보고 늘린 것으로,
#     2026 한 시즌만 볼 때는 안 보이던 문제들이다 (2026-08-29):
#       1차: (팀명 원본, 이름, 등번호) 그대로 일치
#       2차: (괄호 제거한 팀명, 이름, 등번호) 일치
#       3차: (이름, 등번호)만으로 매칭 — 로스터 전체에서 유일할 때만 사용
#       4차: (팀명 원본, 이름)만 — 등번호가 없거나 안 맞을 때, 팀 내 유일할 때만
#       5차: (괄호 제거 팀명, 이름)만 — "GD챌린저스BC(U18)" 처럼 괄호 표기가 다른데
#            등번호까지 어긋나는 경우. 4차로는 안 붙었다.
#     보정 ① 팀명 자동 연결: 명단은 KBSA 가 '지금' 쓰는 팀 이름인데 기록지는 그
#            시즌 이름이다(동산정보산업고 vs 서울동산고, 밀성고 vs TKBC(U19),
#            은평구BC(U19) vs EPBC(U-18)). 별칭표를 손으로 관리하면 매년 놓치므로
#            선수 명부 겹침으로 자동으로 잇는다.
#     보정 ② 개명: 선수가 이름을 바꾸면 명단은 새 이름, 기록지는 옛 이름이다.
#            players_*.xlsx 를 전부 읽어 person_no 별 이름 이력을 만들고, 1~5차로
#            못 붙은 것만 옛 이름으로 한 번 더 본다. 2020~2026 에서 76명이 걸린다.
#            한 명이 수백 행이라 영향이 크다(2021 투수 미매칭 469행 중 327행이 2명).
#     이 보정들을 넣기 전/후로 2020~2025 전 구간 미매칭이 1,600여 행 -> 13행이 됐다.
#   - Streamlit에는 person_no를 노출하지 않고 이름만 보여줄 것이므로,
#     season_full의 batter/pitcher 컬럼은 이름만 남긴 깨끗한 값으로 덮어쓴다
#   - Streamlit이 세부 스탯(OPS/BABIP/wOBA/RC/wRC/wRC+/FIP/RE24 등) 계산에
#     바로 쓸 수 있도록 시즌 전체 play-by-play를 하나로 저장한다
# ════════════════════════════════════════════════════════════════
import re as _re

def _split_name_no(value):
    """"홍길동(10)" -> ("홍길동", "10"). "홍길동()" -> ("홍길동", None) (등번호 미기재).
    닫는 괄호가 오타로 깨져도(예: "이원강(31_") 숫자만 있으면 살려낸다.
    괄호 자체가 없으면 (원문, None)."""
    if pd.isna(value):
        return value, None
    s = str(value).strip()
    m = _re.match(r'^(.*?)\((\d*)\D*$', s)
    if m:
        no = m.group(2) or None
        return m.group(1).strip(), no
    return s, None


def _norm_team(v):
    """"GD챌린저스BC(U-18)" -> "GD챌린저스BC" 처럼 괄호 안 내용을 제거해 비교용으로 정규화."""
    if pd.isna(v):
        return v
    return _re.sub(r'\(.*?\)', '', str(v)).strip()


def _make_parquet_safe(df):
    """object 컬럼에 숫자/문자가 섞여 있으면(예: order 컬럼에 잘못 들어간 문자 하나)
    pyarrow가 타입을 못 정해 to_parquet이 실패한다. NaN은 그대로 두고 나머지 값만
    전부 문자열로 통일해서 저장한다 (xlsx 쪽은 원본 그대로 유지, 이건 parquet 저장용 복사본에만 적용)."""
    df = df.copy()
    obj_cols = df.select_dtypes(include="object").columns
    for col in obj_cols:
        df[col] = df[col].apply(lambda v: v if pd.isna(v) else str(v))
    return df



def export_season_full(year, full, players, name_col, no_col, team_col, person_col):
    """그 해 season_full 을 만들어 xlsx/parquet 로 저장하고 돌려준다.

    예전에는 이 셀 전체가 전역 YEAR/full/players 를 그대로 읽는 통짜 코드였다.
    함수로 묶은 건 **여러 시즌을 한 번에 돌리기 위해서다** — 맨 아래 '여러 연도
    일괄 실행' 셀이 연도마다 이 함수를 부른다. 한 해만 볼 때는 바로 아래 호출이
    예전과 똑같이 동작한다.
    """
    SEASON_FULL_OUT = f"{BASE_DIR}/season_full/season_full_{year}.xlsx"
    # Streamlit은 이 parquet 파일을 우선 읽는다 (xlsx보다 다운로드+파싱이 훨씬 빠름).
    # 원본 파일이 계속 추가되어 이 셀을 반복 재실행해도, xlsx/parquet 둘 다 매번 같이 갱신된다.
    SEASON_FULL_PARQUET_OUT = f"{BASE_DIR}/season_full/season_full_{year}.parquet"
    os.makedirs(os.path.dirname(SEASON_FULL_OUT), exist_ok=True)




    season_full = full.copy()
    season_full['pitcher'] = (
        season_full.groupby(['game_id', 'def_team'])['pitcher']
        .transform(lambda s: s.ffill())
    )

    _batter_split  = season_full['batter'].apply(_split_name_no)
    _pitcher_split = season_full['pitcher'].apply(_split_name_no)
    season_full['batter_no']  = _batter_split.apply(lambda t: t[1])
    season_full['pitcher_no'] = _pitcher_split.apply(lambda t: t[1])
    season_full['batter']  = _batter_split.apply(lambda t: t[0])
    season_full['pitcher'] = _pitcher_split.apply(lambda t: t[0])

    # order는 타순(1~9) 숫자인데 원본 스코어시트에 가끔 오타 문자가 섞여 있다.
    # 문자열로 그대로 두면 parquet 저장은 되지만(_make_parquet_safe), Streamlit의
    # 득점(R) 계산이 order를 int로 비교하기 때문에 숫자로 정리해둬야 한다.
    season_full['order'] = pd.to_numeric(season_full['order'], errors='coerce')

    if None in (name_col, no_col, team_col, person_col):
        print("⚠ 위 진단 셀에서 컬럼 매칭이 실패한 상태라 person_no를 붙이지 못합니다.")
        print("  진단 셀의 _NAME_CANDS/_NO_CANDS/_TEAM_CANDS/_PERSON_CANDS 를 고치고 두 셀을 다시 실행하세요.")
        season_full['batter_person_no']  = None
        season_full['pitcher_person_no'] = None
    else:
        import glob as _glob

        players = players.copy()
        players[no_col] = players[no_col].astype(str).str.extract(r'(\d+)')[0]
        players['_team_norm'] = players[team_col].apply(_norm_team)

        def _build_maps(df):
            """1~5차 조회표를 한 벌 만든다. 등번호를 무시하는 3~5차는 후보가 유일할
            때만 담는다 — 유일하지 않으면 붙이는 게 아니라 비워 둬야 동명이인을
            조용히 한쪽으로 몰지 않는다."""
            if df.empty:
                return None
            u_name_no   = df.groupby([name_col, no_col]).filter(lambda d: d[person_col].nunique() == 1)
            u_team_name = df.groupby([team_col, name_col]).filter(lambda d: d[person_col].nunique() == 1)
            u_norm_name = df.groupby(['_team_norm', name_col]).filter(lambda d: d[person_col].nunique() == 1)
            return {
                'raw':  df.set_index([team_col, name_col, no_col])[person_col].to_dict(),
                'norm': df.set_index(['_team_norm', name_col, no_col])[person_col].to_dict(),
                'name_no':   u_name_no.set_index([name_col, no_col])[person_col].to_dict() if not u_name_no.empty else {},
                'team_name': u_team_name.set_index([team_col, name_col])[person_col].to_dict() if not u_team_name.empty else {},
                'norm_name': u_norm_name.set_index(['_team_norm', name_col])[person_col].to_dict() if not u_norm_name.empty else {},
            }

        _maps = _build_maps(players)

        # ── 보정 ② 개명: person_no 별 '다른 시즌에 쓰던 이름' 사전 ──────────
        # 명단은 개명하면 새 이름으로만 남는데 기록지는 그 시즌 이름 그대로다.
        # 여러 시즌 명단을 겹쳐야만 드러난다: 201407000836 = 2020 문준서 -> 2021 문승현.
        _hist = {}
        _roster_files = sorted(_glob.glob(f"{BASE_DIR}/players_by_season/players_*.xlsx"))
        for _p in _roster_files:
            try:
                _d = pd.read_excel(_p, usecols=[person_col, name_col])
            except Exception as _e:
                print(f"  (건너뜀) {_p}: {_e}")
                continue
            for _pno, _nm in zip(_d[person_col], _d[name_col]):
                if pd.notna(_pno) and pd.notna(_nm):
                    _hist.setdefault(str(_pno), set()).add(str(_nm).strip())
        _hist = {k: v for k, v in _hist.items() if len(v) > 1}
        print(f"개명 대조: 명단 {len(_roster_files)}개 파일 -> 이름이 바뀐 person_no {len(_hist)}명")

        _alt_rows = []
        for _rec in players.to_dict('records'):
            for _other in _hist.get(str(_rec[person_col]), ()):
                if _other != _rec[name_col]:
                    _alt_rows.append({**_rec, name_col: _other})
        _alt_maps = _build_maps(pd.DataFrame(_alt_rows)) if _alt_rows else None

        # ── 보정 ① 팀명 자동 연결 ────────────────────────────────────────
        # 기록지 팀명이 명단에 없으면, 같은 (이름, 등번호)를 몇 명이나 공유하는지로
        # 같은 팀인지 판정한다. 다른 팀끼리 명부가 절반 넘게 겹치는 일은 없다.
        def _members_in_season_full():
            out = {}
            for _tc, _nc, _oc in (('team', 'batter', 'batter_no'),
                                  ('def_team', 'pitcher', 'pitcher_no')):
                _sub = season_full[[_tc, _nc, _oc]].dropna(subset=[_nc])
                for _t, _nm, _no in _sub.itertuples(index=False):
                    if pd.isna(_t):
                        continue
                    out.setdefault(_t, set()).add((_nm, _no or ''))
            return out

        _roster_members = {t: set(zip(d[name_col], d[no_col].fillna('')))
                           for t, d in players.groupby(team_col)}
        _known_teams = set(_roster_members) | {_norm_team(t) for t in _roster_members}
        TEAM_ALIAS = {}
        for _t, _mem in _members_in_season_full().items():
            if _t in _known_teams or _norm_team(_t) in _known_teams or not _mem:
                continue
            _scored = sorted(((len(_mem & m), rt) for rt, m in _roster_members.items()), reverse=True)
            _best   = _scored[0] if _scored else (0, None)
            _second = _scored[1] if len(_scored) > 1 else (0, None)
            # 겹침 3명 이상 + 그 팀 인원의 50% 이상 + 2등의 2배 이상일 때만 잇는다.
            if _best[0] >= 3 and _best[0] >= 0.5 * len(_mem) and _best[0] >= 2 * _second[0]:
                TEAM_ALIAS[_t] = _best[1]
        if TEAM_ALIAS:
            print("기록지<->명단 팀명 자동 연결:",
                  ", ".join(f"{k}->{v}" for k, v in sorted(TEAM_ALIAS.items())))
        else:
            print("기록지<->명단 팀명 자동 연결: 없음 (표기가 모두 일치)")

        def _lookup_in(maps, team, nteam, name, no):
            if maps is None:
                return None, None
            if (team, name, no) in maps['raw']:
                return maps['raw'][(team, name, no)], 'team(원본)+name+no'
            if (nteam, name, no) in maps['norm']:
                return maps['norm'][(nteam, name, no)], 'team(괄호제거)+name+no'
            if (name, no) in maps['name_no']:
                return maps['name_no'][(name, no)], 'name+no(팀 무시, 로스터 내 유일)'
            if (team, name) in maps['team_name']:
                return maps['team_name'][(team, name)], 'team+name(등번호 무시, 팀 내 유일)'
            if (nteam, name) in maps['norm_name']:
                return maps['norm_name'][(nteam, name)], 'team(괄호제거)+name(등번호 무시)'
            return None, None

        def _lookup(team, name, no):
            team = TEAM_ALIAS.get(team, team)
            nteam = _norm_team(team)
            pno, via = _lookup_in(_maps, team, nteam, name, no)
            if pno is not None or _alt_maps is None:
                return pno, via
            # 현재 이름으로 못 붙은 것만 '옛 이름' 판에서 한 번 더 본다.
            pno, via = _lookup_in(_alt_maps, team, nteam, name, no)
            return pno, (f'개명({via})' if via else None)

        def _attach(tcol, ncol, ocol, out_col, via_col):
            """같은 (팀, 이름, 번호)가 수만 행 반복되므로 고유 조합에만 조회를 돌린다.
            예전에는 apply(axis=1)로 7만 행마다 조회해 몇 분씩 걸렸다."""
            _keys = season_full[[tcol, ncol, ocol]].dropna(subset=[ncol]).drop_duplicates()
            _resolved = {tuple(k): _lookup(*k) for k in _keys.itertuples(index=False)}
            _trio = list(zip(season_full[tcol], season_full[ncol], season_full[ocol]))
            season_full[out_col] = [_resolved.get(t, (None, None))[0] for t in _trio]
            season_full[via_col] = [_resolved.get(t, (None, None))[1] for t in _trio]

        _attach('team', 'batter', 'batter_no', 'batter_person_no', '_batter_match_via')
        _attach('def_team', 'pitcher', 'pitcher_no', 'pitcher_person_no', '_pitcher_match_via')

        from collections import Counter
        batter_has  = season_full['batter'].notna()
        pitcher_has = season_full['pitcher'].notna()
        print("batter person_no 매칭 방식 분포:", dict(Counter(season_full.loc[batter_has, '_batter_match_via'])))
        print("pitcher person_no 매칭 방식 분포:", dict(Counter(season_full.loc[pitcher_has, '_pitcher_match_via'])))

        missing_batter  = batter_has  & season_full['batter_person_no'].isna()
        missing_pitcher = pitcher_has & season_full['pitcher_person_no'].isna()
        print(f"batter person_no 매칭 실패: {missing_batter.sum()}행")
        print(f"pitcher person_no 매칭 실패: {missing_pitcher.sum()}행")
        if missing_batter.sum():
            sample = season_full.loc[missing_batter, ['batter', 'batter_no', 'team']].drop_duplicates().head(10)
            print("  예시(batter, batter_no, team):", sample.values.tolist())
        if missing_pitcher.sum():
            sample = season_full.loc[missing_pitcher, ['pitcher', 'pitcher_no', 'def_team']].drop_duplicates().head(10)
            print("  예시(pitcher, pitcher_no, def_team):", sample.values.tolist())

        season_full = season_full.drop(columns=['_batter_match_via', '_pitcher_match_via'])

    season_full.to_excel(SEASON_FULL_OUT, index=False)


    _make_parquet_safe(season_full).to_parquet(SEASON_FULL_PARQUET_OUT, index=False)
    print(f"\n완료: season_full(xlsx, 수기 확인용) 저장 -> {SEASON_FULL_OUT} ({len(season_full)}행)")
    print(f"완료: season_full(parquet, Streamlit이 실제로 읽는 파일) 저장 -> {SEASON_FULL_PARQUET_OUT}")


    return season_full


# 한 해만 돌릴 때(예전과 같은 동작). 여러 해는 맨 아래 일괄 실행 셀을 쓴다.
season_full = export_season_full(YEAR, full, players, name_col, no_col, team_col, person_col)

# pitcher_box + players parquet export (Streamlit 로딩 속도 개선)

In [ ]:
# ════════════════════════════════════════════════════════════════
# pitcher_box + players parquet export (Streamlit 로딩 속도 개선)
#   - season_full과 같은 이유(xlsx 대신 parquet를 읽으면 다운로드+파싱이 훨씬 빠름)로
#     이 둘도 parquet로 같이 내보낸다.
#   - 위쪽 "등번호 000 채우기"나 person_no 매칭 셀들과 무관하게, 그 시점의 실제 xlsx
#     파일을 새로 읽어서 만든다 — 그래서 이 셀만 따로 실행해도 항상 최신 상태로 동기화된다.
#   - _make_parquet_safe()는 "시즌 통합 파일 export" 셀에서 정의한 함수를 재사용한다
#     (object 컬럼에 타입이 섞여 있으면 pyarrow가 실패하는 걸 막기 위해 문자열로 통일).
# ════════════════════════════════════════════════════════════════
def export_side_parquets(year):
    """pitcher_box_{year} / players_{year} 를 parquet 로도 저장한다."""
    box_xlsx = f"{BASE_DIR}/pitcher_box/pitcher_box_{year}.xlsx"
    box_out  = f"{BASE_DIR}/pitcher_box/pitcher_box_{year}.parquet"
    ply_xlsx = f"{BASE_DIR}/players_by_season/players_{year}.xlsx"
    ply_out  = f"{BASE_DIR}/players_by_season/players_{year}.parquet"

    if os.path.exists(box_xlsx):
        box = pd.read_excel(box_xlsx)
        _make_parquet_safe(box).to_parquet(box_out, index=False)
        print(f"완료: pitcher_box(parquet) 저장 -> {box_out} ({len(box)}행)")
    else:
        print(f"⚠ {box_xlsx} 가 없습니다 — 박스스코어 크롤러를 먼저 돌려야 합니다.")

    ply = pd.read_excel(ply_xlsx)
    _make_parquet_safe(ply).to_parquet(ply_out, index=False)
    print(f"완료: players(parquet) 저장 -> {ply_out} ({len(ply)}행)")


export_side_parquets(YEAR)


In [ ]:
# ════════════════════════════════════════════════════════════════
# person_no 매칭 실패 목록 export (players_{year}.xlsx 로스터 수기 보정용)
#   - export 셀에서 5단계 + 팀명 자동 연결 + 개명 보정을 다 거쳐도 person_no가 안 붙은
#     행들을 선수(팀+이름+등번호)별로 묶어서 엑셀로 뽑는다
#   - 발생행수/game_id/file_name 예시를 같이 넣어서, 실제 스코어시트에서 왜 안 맞는지
#     (오타/미기재/시즌 중 변경 등) 바로 찾아갈 수 있게 한다
#   - "진단" 컬럼: players 로스터에서 (팀,이름)으로 후보를 찾아 0명이면 "로스터에 없음",
#     1명이면 "로스터엔 유일"(원본 등번호 표기 문제일 가능성), 2명 이상이면 "동명이인"
#     으로 자동 분류한다
#   - 2020~2026 실측 기준 여기 남는 건 대부분 기록지 오타이거나 KBSA 미등록이다
# ════════════════════════════════════════════════════════════════
def report_match_failures(year, season_full, players, name_col, no_col, team_col):
    """미매칭 선수 목록을 {year}_person_no_매칭실패.xlsx 로 저장한다."""
    out_path = f"{BASE_DIR}/season_full/{year}_person_no_매칭실패.xlsx"

    def diagnose(team, name):
        cands = players[(players[team_col] == team) & (players[name_col] == name)]
        n = len(cands)
        if n == 0:
            return "로스터에 없음"
        nos = cands[no_col].apply(
            lambda v: '빈값' if pd.isna(v) or str(v).strip() in ('', 'nan') else str(v)).tolist()
        if n == 1:
            return f"로스터엔 유일(등번호={nos[0]}) — 원본 등번호 표기 문제일 가능성"
        return f"동명이인 {n}명(등번호: {', '.join(nos)}) — 실제 확인 필요"

    def summarize(mask, name_c, no_c, team_c):
        cols = [c for c in [team_c, name_c, no_c, 'game_id', 'file_name'] if c in season_full.columns]
        sub = season_full.loc[mask, cols]
        if sub.empty:
            return pd.DataFrame(columns=[team_c, name_c, no_c, '발생행수', 'game_id_예시', 'file_name_예시', '진단'])
        agg = {
            '발생행수': ('game_id', 'size'),
            'game_id_예시': ('game_id', lambda s: sorted(s.dropna().unique().tolist())[:5]),
        }
        if 'file_name' in sub.columns:
            agg['file_name_예시'] = ('file_name', lambda s: sorted(s.dropna().unique().tolist())[:5])
        result = (sub.groupby([team_c, name_c, no_c], dropna=False)
                  .agg(**agg).reset_index().sort_values('발생행수', ascending=False))
        result['진단'] = result.apply(lambda r: diagnose(r[team_c], r[name_c]), axis=1)
        return result

    miss_bat = season_full['batter'].notna() & season_full['batter_person_no'].isna()
    miss_pit = season_full['pitcher'].notna() & season_full['pitcher_person_no'].isna()
    fail_bat = summarize(miss_bat, 'batter', 'batter_no', 'team')
    fail_pit = summarize(miss_pit, 'pitcher', 'pitcher_no', 'def_team')

    with pd.ExcelWriter(out_path) as writer:
        fail_bat.to_excel(writer, sheet_name="batter_매칭실패", index=False)
        fail_pit.to_excel(writer, sheet_name="pitcher_매칭실패", index=False)

    print(f"타자 매칭 실패 선수(고유 팀+이름+등번호 조합): {len(fail_bat)}명")
    print(f"투수 매칭 실패 선수(고유 팀+이름+등번호 조합): {len(fail_pit)}명")
    if len(fail_bat):
        print(fail_bat[['team', 'batter', 'batter_no', '발생행수', '진단']].to_string(index=False))
    if len(fail_pit):
        print(fail_pit[['def_team', 'pitcher', 'pitcher_no', '발생행수', '진단']].to_string(index=False))
    print(f"저장 -> {out_path}")
    return fail_bat, fail_pit


report_match_failures(YEAR, season_full, players, name_col, no_col, team_col)


In [ ]:
# ════════════════════════════════════════════════════════════════
# 등번호가 "(0)" 또는 "()"(미기재)인 행 전부 export (원본 스코어시트 수기 보정용)
#   - person_no 매칭 성공 여부와 상관없이, batter_no/pitcher_no가 비어있거나 "0"인
#     모든 행을 찾는다 (매칭은 팀+이름 폴백으로 성공했더라도 원본에는 비어있을 수 있음)
#   - 같은 선수가 한 경기에서 여러 타석에 나오면 중복이라 (file_name, game_id, team,
#     이름, 번호) 조합으로 묶어서 한 행씩만 남긴다
#   - file_name까지 나오므로 원본 엑셀에서 바로 찾아 고칠 수 있다
# ════════════════════════════════════════════════════════════════
def report_missing_numbers(year, season_full):
    """등번호가 비었거나 0인 행 목록을 {year}_등번호_누락또는0.xlsx 로 저장한다."""
    out_path = f"{BASE_DIR}/season_full/{year}_등번호_누락또는0.xlsx"

    def missing(s):
        return s.isna() | (s.astype(str).str.strip() == '0')

    bat_mask = season_full['batter'].notna() & missing(season_full['batter_no'])
    pit_mask = season_full['pitcher'].notna() & missing(season_full['pitcher_no'])

    bat_cols = [c for c in ['file_name', 'game_id', 'team', 'batter', 'batter_no']
                if c in season_full.columns]
    pit_cols = [c for c in ['file_name', 'game_id', 'def_team', 'pitcher', 'pitcher_no']
                if c in season_full.columns]

    bat = season_full.loc[bat_mask, bat_cols].drop_duplicates().sort_values(bat_cols[:2])
    pit = season_full.loc[pit_mask, pit_cols].drop_duplicates().sort_values(pit_cols[:2])

    with pd.ExcelWriter(out_path) as writer:
        bat.to_excel(writer, sheet_name="batter_등번호누락", index=False)
        pit.to_excel(writer, sheet_name="pitcher_등번호누락", index=False)

    print(f"타자 등번호 누락/0 (파일+경기+선수 조합): {len(bat)}건")
    print(f"투수 등번호 누락/0 (파일+경기+선수 조합): {len(pit)}건")
    print(f"저장 -> {out_path}")
    return bat, pit


report_missing_numbers(YEAR, season_full)


In [ ]:
# ════════════════════════════════════════════════════════════════
# schedule_{year}.xlsx에서 이미 열린(=full에 들어간) 경기 정리
#   - "경기 일정 크롤링.ipynb"는 아직 안 열린 경기를 새로 찾아서 채우는 역할이라
#     네트워크 요청이 많아 느리다(경기당 9~11초). 반대로 "이미 열린 경기를
#     예정 목록에서 빼는 것"은 이미 메모리에 있는 full의 game_id와 대조만
#     하면 되는 순수 로컬 연산이라 네트워크 요청이 전혀 없다 — 그래서 이 셀은
#     빠르고, season_full을 새로 뽑을 때마다 매번 같이 돌려도 부담 없다.
#   - 즉 "새 예정 경기 추가"는 경기 일정 크롤링.ipynb가, "이미 지난 예정 경기
#     제거"는 이 셀이 나눠서 맡는다.
#   - 파일을 고치는 유일한 자동 셀이지만, 하는 일이 "이미 열린 경기를 예정 목록에서
#     빼는 것"뿐이라 몇 번을 돌려도 결과가 같다. 지난 시즌은 schedule 파일 자체가
#     없어 그냥 건너뛴다.
# ════════════════════════════════════════════════════════════════
def clean_schedule(year, full):
    """그 해 예정 목록에서 이미 치른 경기를 지운다."""
    path = f"{BASE_DIR}/season_full/schedule_{year}.xlsx"
    if not os.path.exists(path):
        print(f"⚠ {path} 가 없습니다 — '경기 일정 크롤링.ipynb'를 먼저 실행하면 생깁니다.")
        return

    schedule_df = pd.read_excel(path, dtype={"game_id": str})
    # 예정 경기가 하나도 안 남으면 시트가 통째로 비어(헤더도 없이) 저장될 수 있다.
    # 그 상태로 game_id 를 집으면 KeyError 라, 지울 게 없는 정상 상태로 보고 넘어간다.
    if schedule_df.empty or "game_id" not in schedule_df.columns:
        print(f"예정 경기 없음 — schedule 이 비어 있습니다 ({path})")
        return

    full_ids = set(pd.to_numeric(full["game_id"], errors="coerce").dropna().astype(int).tolist())
    sched_ids = pd.to_numeric(schedule_df["game_id"], errors="coerce")
    still_upcoming = schedule_df[~sched_ids.isin(full_ids) | sched_ids.isna()]

    removed = len(schedule_df) - len(still_upcoming)
    if removed:
        still_upcoming.to_excel(path, index=False)
        print(f"이미 열린 경기 {removed}건을 schedule에서 제거 -> {path} ({len(still_upcoming)}행 남음)")
    else:
        print("제거할 경기 없음(schedule에 있는 경기 전부 아직 full에 없음)")


clean_schedule(YEAR, full)


# 여러 연도 일괄 실행

In [ ]:
# ════════════════════════════════════════════════════════════════
# 여러 연도 일괄 실행
#   - 맨 위 YEARS 에 적은 시즌을 순서대로 전부 처리한다.
#       YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
#     처럼 적고 이 셀만 돌리면 season_full 이 연도별로 다 만들어진다.
#   - 한 해당 하는 일은 위 셀들과 완전히 같다(같은 함수를 부른다):
#       full 로드 -> 특수경기결과 보충 -> 명단 컬럼 매칭 -> season_full 저장(xlsx+parquet)
#       -> pitcher_box/players parquet -> 매칭실패 목록 -> 등번호 누락 목록
#       -> 예정 일정에서 이미 치른 경기 제거
#   - 한 해가 실패해도 나머지는 계속 돈다(원본이 아직 없는 해가 섞여 있어도 멈추지 않게).
#     실패한 연도는 맨 끝에 모아서 찍는다.
#   - 마지막으로 처리한 연도의 full/players/season_full 이 전역에 남으므로,
#     끝난 뒤 아래 수동 점검 셀들을 그대로 이어서 쓸 수 있다.
#
#   ※ 이 셀은 원본을 고치지 않는다. '등번호 000 채우기'와 '개명 반영'은 DRY_RUN 이
#     달린 수동 셀이라 일부러 여기 넣지 않았다.
# ════════════════════════════════════════════════════════════════
SKIP_EXISTING = False   # True 면 season_full_{year}.parquet 이 이미 있는 해는 건너뛴다

_done, _failed = [], []

for _year in YEARS:
    print(f"\n{'=' * 60}\n{_year} 시즌\n{'=' * 60}")
    _out_parquet = f"{BASE_DIR}/season_full/season_full_{_year}.parquet"
    if SKIP_EXISTING and os.path.exists(_out_parquet):
        print(f"이미 있음 — 건너뜀 ({_out_parquet})")
        continue
    try:
        full = load_full(_year)
        full = add_special_games(full, _year)
        players, name_col, no_col, team_col, person_col = match_player_columns(_year, full)
        season_full = export_season_full(
            _year, full, players, name_col, no_col, team_col, person_col)
        export_side_parquets(_year)
        report_match_failures(_year, season_full, players, name_col, no_col, team_col)
        report_missing_numbers(_year, season_full)
        clean_schedule(_year, full)
        YEAR = _year
        _done.append(_year)
    except Exception as _e:
        # 한 해가 넘어져도 나머지는 돌린다 — 원본이 아직 없는 해가 섞여 있을 수 있다.
        print(f"❌ {_year} 실패: {type(_e).__name__}: {_e}")
        _failed.append((_year, f"{type(_e).__name__}: {_e}"))

print(f"\n{'=' * 60}")
print(f"완료 {len(_done)}개 시즌: {_done}")
if _failed:
    print(f"실패 {len(_failed)}개:")
    for _y, _msg in _failed:
        print(f"  {_y}: {_msg}")


In [ ]:
# ════════════════════════════════════════════════════════════════
# 원본 스코어시트 선수 개명 반영 (players_{YEAR}.xlsx엔 새 이름으로 등록돼
# 있는데 원본 스코어시트는 옛 이름 그대로라 매칭이 실패하는 경우):
#   - 라온고 유강빈 -> 유강현 (등번호 2, 로스터와 이미 일치 — 이름만 바꾸면 됨)
#
#   DRY_RUN=True: 실제로 저장하지 않고 어떤 파일의 몇 행이 바뀌는지만 보여준다.
#   확인 후 DRY_RUN=False로 바꿔서 다시 실행해야 원본 파일에 실제로 저장된다.
# ════════════════════════════════════════════════════════════════
DRY_RUN = True  # 확인 후 False로 바꿔서 재실행 -> 실제 원본 파일에 저장

RENAME_TARGETS = [
    # (season_full 컬럼, season_full 팀 컬럼, 팀, 원본(옛) 이름, 새 이름, 등번호)
    ('pitcher', 'def_team', '라온고', '유강빈', '유강현', '2'),
]


def _rename_with_no(value, old_name, new_name, correct_no):
    """"유강빈(2)" -> "유강현(2)". old_name과 다른 이름이면 그대로 둔다."""
    if pd.isna(value):
        return value, False
    s = str(value).strip()
    m = _re5.match(r'^(.*?)\((\d*)\D*$', s)
    if not m or m.group(1).strip() != old_name:
        return value, False
    return f"{new_name}({correct_no})", True


total_renamed = 0
for col, team_col_sf, team, old_name, new_name, correct_no in RENAME_TARGETS:
    affected_files = sorted(
        season_full.loc[(season_full[team_col_sf] == team) & (season_full[col] == old_name), 'file_name']
        .dropna().unique().tolist()
    )
    print(f"\n=== {team} {old_name} -> {new_name}({correct_no}번), {col} 컬럼 ({len(affected_files)}개 파일) ===")
    for fname in affected_files:
        fpath = _file_locations.get(fname)
        if fpath is None:
            print(f"  ⚠ {fname}.xlsx 위치를 못 찾음 — 건너뜀")
            continue
        raw_df = pd.read_excel(fpath)
        if col not in raw_df.columns:
            print(f"  ⚠ {fname}.xlsx에 {col} 컬럼 없음 — 건너뜀")
            continue

        # pitcher 컬럼은 이닝 내내 값이 유지되고 team/def_team은 실제 타석 행에서만
        # 채워지는 등 채움 방식이 batter/team과 달라서(동시에 비지 않음), 팀으로
        # 한 번 더 거르지 않고 이 파일 안의 old_name을 전부 바꾼다 — 이미
        # season_full에서 이 파일이 이 팀+이름 조합을 포함한다고 확인된 상태다.
        new_col_vals = []
        changed_rows = 0
        for cell_val in raw_df[col]:
            new_val, changed = _rename_with_no(cell_val, old_name, new_name, correct_no)
            new_col_vals.append(new_val)
            if changed:
                changed_rows += 1

        if changed_rows == 0:
            continue
        print(f"  {fname}.xlsx: {changed_rows}행 변경")
        total_renamed += changed_rows
        if not DRY_RUN:
            raw_df[col] = new_col_vals
            raw_df.to_excel(fpath, index=False)

print(f"\n{'[DRY RUN] ' if DRY_RUN else ''}총 {total_renamed}행 변경 {'예정' if DRY_RUN else '완료'}")
if DRY_RUN:
    print("실제로 반영하려면 위 DRY_RUN = False 로 바꾸고 이 셀을 다시 실행하세요.")
else:
    print("원본 파일에 저장 완료 — 이제 위쪽 '시즌 통합 파일 export' 셀부터 다시 실행해서")
    print("season_full/person_no 매칭이 개선됐는지 확인하세요.")
